# Séance 9 – Héritage et polymorphisme
### Lab : hiérarchies de classes, mixins et classes abstraites

| | |
|---|---|
| **Durée estimée** | 60 min (2 × 30 min) + bonus |
| **Projet fil rouge** | GestParc – étape 6 : hiérarchie d'équipements |

**Objectifs**
- Créer des sous-classes avec super() et redéfinir des méthodes
- Comprendre le MRO et utiliser des mixins
- Exploiter le polymorphisme et le duck typing
- Imposer un contrat avec une classe abstraite (abc)

**Mode d'emploi**
- Exécuter une cellule : **Maj + Entrée** ; en cas de comportement étrange : *Redémarrer le noyau et tout exécuter*.
- Complétez les cellules marquées `# TODO`, puis exécutez la cellule **✅ Vérification** qui suit : aucune erreur = réponse correcte.
- Difficulté : ⭐ facile · ⭐⭐ intermédiaire · ⭐⭐⭐ avancé · Indices dans les blocs 💡 repliables.

---
## Partie 1 – Héritage simple, super() et redéfinition

> **Rappel**
> ```python
> class Fille(Mere):
>     def __init__(self, a, b, c):
>         super().__init__(a, b)       # initialisation de la partie « mère »
>         self.c = c                   # attribut propre à la fille
>
>     def methode(self):               # redéfinition
>         resultat = super().methode() # extension (facultative)
>         return ...
> ```
> - `isinstance(obj, Mere)` est vrai pour une instance de `Fille`

### Exemple guidé : comptes utilisateurs

In [ ]:
class UserAccount:
    """Compte utilisateur standard."""

    def __init__(self, login: str) -> None:
        self.login = login
        self.groups: list[str] = ["users"]

    def can(self, action: str) -> bool:
        return action in {"read", "write"}

    def __str__(self) -> str:
        return f"{self.login} {self.groups}"


class AdminAccount(UserAccount):
    """Compte administrateur : droits étendus."""

    def __init__(self, login: str, scope: str) -> None:
        super().__init__(login)
        self.scope = scope
        self.groups.append("admins")

    def can(self, action: str) -> bool:
        return super().can(action) or action in {"delete", "configure"}


alice = AdminAccount("alice", scope="réseau")
print(alice, alice.can("read"), alice.can("delete"), UserAccount("bob").can("delete"))
print(isinstance(alice, UserAccount), AdminAccount.__bases__)

### Exercice 1 ⭐ – Manager hérite d'Employee

La classe `Employee` est fournie. Écrivez `Manager(Employee)` :
- paramètre et attribut supplémentaires : `team` (liste de `Employee`, vide par défaut — attention au piège !) ;
- `add_member(employee)` ;
- `__str__` **étend** celui d'`Employee` : `"Sonia Haddad (DSI) – manager de 2 personne(s)"`.

In [ ]:
class Employee:
    """Employé."""

    def __init__(self, first_name: str, last_name: str, department: str) -> None:
        self.first_name = first_name
        self.last_name = last_name
        self.department = department

    def __str__(self) -> str:
        return f"{self.first_name} {self.last_name} ({self.department})"


class Manager(Employee):
    """Employé qui encadre une équipe."""
    # TODO


sonia = Manager("Sonia", "Haddad", "DSI")
sonia.add_member(Employee("Amira", "Ben Salah", "DSI"))
sonia.add_member(Employee("Youssef", "Gharbi", "DSI"))
print(sonia)

<details><summary>💡 Indice</summary>

Commencez `__init__` par `super().__init__(first_name, last_name, department)`. Dans `__str__`, réutilisez `super().__str__()`.

</details>

In [ ]:
# ✅ Vérification
m1, m2 = Manager("Sonia", "Haddad", "DSI"), Manager("Karim", "Trabelsi", "Finance")
m1.add_member(Employee("A", "B", "DSI"))
assert m2.team == [], "les équipes sont partagées !"
assert str(m1) == "Sonia Haddad (DSI) – manager de 1 personne(s)"
assert isinstance(m1, Employee) and m1.department == "DSI"
print("✅ Exercice 1 réussi")

### Exercice 2 ⭐⭐ – Notifications e-mail et SMS

La classe `Notification` formate un message (`format()`) et l'« envoie » (`send()` renvoie le texte transmis).
Écrivez :
- `EmailNotification(recipient, subject, body)` : `format()` **étend** celui du parent en ajoutant une ligne `Objet : …` au début ;
  `send()` renvoie `"EMAIL à … : <message formaté>"` ;
- `SmsNotification(phone, body)` : `format()` **redéfinit** le parent : message limité à 30 caractères (`…` final si tronqué) ;
  `send()` renvoie `"SMS à … : <message>"`.

In [ ]:
class Notification:
    """Notification générique."""

    def __init__(self, recipient: str, body: str, level: str = "INFO") -> None:
        self.recipient = recipient
        self.body = body
        self.level = level

    def format(self) -> str:
        return f"[{self.level}] {self.body}"

    def send(self) -> str:
        return f"À {self.recipient} : {self.format()}"


# TODO : EmailNotification et SmsNotification


alerts = [
    EmailNotification("dsi@entreprise.tn", "Disque plein", "srv-db-01 : /var à 97 %", level="CRITICAL"),
    SmsNotification("+216 20 000 000", "srv-db-01 : /var à 97 %, intervention requise", level="CRITICAL"),
]
for alert in alerts:
    print(alert.send())

<details><summary>💡 Indice</summary>

`SmsNotification` n'a pas besoin de `__init__` : elle hérite de celui du parent. Pour tronquer : `text[:29] + "…"`.

</details>

In [ ]:
# ✅ Vérification
e = EmailNotification("a@b.tn", "Test", "corps")
assert e.format() == "Objet : Test\n[INFO] corps"
assert e.send() == "EMAIL à a@b.tn : Objet : Test\n[INFO] corps"
s = SmsNotification("+216", "message très long qui dépasse la limite")
assert s.format() == "[INFO] message très long qui …" and len(s.format()) == 30
assert SmsNotification("+216", "court").send() == "SMS à +216 : [INFO] court"
assert isinstance(s, Notification) and s.level == "INFO"
print("✅ Exercice 2 réussi")

### Exercice 3 ⭐⭐ – Hiérarchie de personnages du moteur de jeu

On reprend le TP de S8 : `Player` et `Enemy` dupliquaient `attack`, `defense`, `strike`… La classe mère `Character` est fournie.
Écrivez :
- `Player(name)` : 30 PV, attaque 5, défense 2 ; attributs `xp = 0` et `inventory` ; méthode `drink(potion_value)` qui soigne ;
- `Enemy(name, max_hp, attack, defense, xp)` ;
- `Boss(Enemy)` : attribut `max_hp` hérité ; `strike` est **redéfinie** : si le boss a perdu plus de la moitié de ses PV, il est enragé (`enraged` vaut `True`) et frappe avec +3 d'attaque.
  `__str__` **étend** celui de `Character` en préfixant `"👑 "`.

Simulez ensuite le combat fourni.

In [ ]:
class Character:
    """Personnage générique (fourni)."""

    def __init__(self, name: str, max_hp: int, attack: int, defense: int) -> None:
        self.name = name
        self.max_hp = max_hp
        self.hp = max_hp
        self.attack = attack
        self.defense = defense

    @property
    def is_alive(self) -> bool:
        return self.hp > 0

    def take_damage(self, amount: int) -> int:
        lost = min(amount, self.hp)
        self.hp -= lost
        return lost

    def strike(self, target: "Character") -> int:
        return target.take_damage(max(1, self.attack - target.defense))

    def __str__(self) -> str:
        return f"{self.name} ({self.hp}/{self.max_hp} PV)"


# TODO : Player, Enemy, Boss


hero = Player("Aya")
hero.attack, hero.defense = 12, 4          # épée et bouclier équipés
dragon = Boss("Dragon", max_hp=40, attack=8, defense=3, xp=100)
log = []
while hero.is_alive and dragon.is_alive:
    log.append(f"{hero.name} -> {hero.strike(dragon)}")
    if dragon.is_alive:
        log.append(f"{dragon.name} -> {dragon.strike(hero)}{' (enragé)' if dragon.enraged else ''}")
print(log, hero, dragon, sep="\n")

<details><summary>💡 Indice</summary>

`enraged` peut être une propriété. Dans `Boss.strike`, calculez les dégâts avec le bonus puis appelez `target.take_damage(...)`.

</details>

In [ ]:
# ✅ Vérification
p = Player("Test")
assert (p.hp, p.attack, p.defense, p.xp, p.inventory) == (30, 5, 2, 0, [])
p.take_damage(10)
assert p.drink(25) == 10 and p.hp == 30
b = Boss("B", max_hp=20, attack=8, defense=0, xp=5)
assert isinstance(b, Enemy) and isinstance(b, Character) and b.xp == 5
assert not b.enraged and b.strike(Player("X")) == 6
b.take_damage(11)
assert b.enraged and b.strike(Player("Y")) == 9
assert str(b) == "👑 B (9/20 PV)"
assert not dragon.is_alive and hero.hp == 8
assert log[-1] == "Aya -> 4" and log.count("Dragon -> 7 (enragé)") == 2
print("✅ Exercice 3 réussi")

### Exercice 4 ⭐ – isinstance, issubclass et filtrage par type

Avec les classes de l'exercice 3 :
1. `party` contient des personnages variés. Construisez `enemies` (tous les `Enemy`, boss compris) et `bosses` (uniquement les `Boss`) par compréhension.
2. `total_xp` : somme de l'XP de tous les ennemis.
3. `hierarchy` : la liste des noms des classes du MRO de `Boss` (`Boss.__mro__`).

In [ ]:
party = [Player("Aya"), Enemy("Gobelin", 12, 6, 1, 10), Boss("Dragon", 40, 8, 3, 100),
         Enemy("Rat", 5, 2, 0, 2), Player("Karim")]
# TODO
enemies = ...
bosses = ...
total_xp = ...
hierarchy = ...

<details><summary>💡 Indice</summary>

`type(c) == Enemy` exclurait le boss : utilisez `isinstance`.

</details>

In [ ]:
# ✅ Vérification
assert [e.name for e in enemies] == ["Gobelin", "Dragon", "Rat"]
assert [b.name for b in bosses] == ["Dragon"]
assert total_xp == 112
assert hierarchy == ["Boss", "Enemy", "Character", "object"]
print("✅ Exercice 4 réussi")

---
## Partie 2 – Héritage multiple, mixins, polymorphisme, classes abstraites

> **Rappel**
> - `class D(B, C)` · MRO : `D.__mro__` · `super()` suit le MRO
> - Mixin : petite classe qui ajoute une capacité, placée **avant** la classe principale
> - Polymorphisme : même appel, comportement propre à chaque classe ; duck typing : seule la méthode compte
> - `from abc import ABC, abstractmethod` · une classe avec une méthode abstraite non implémentée ne peut pas être instanciée

### Exemple guidé : des contrôles de conformité polymorphes

In [ ]:
class PasswordPolicyCheck:
    def run(self) -> tuple[bool, str]:
        return True, "longueur minimale 12 caractères"


class FirewallCheck:
    def __init__(self, open_ports: set[int]) -> None:
        self.open_ports = open_ports

    def run(self) -> tuple[bool, str]:
        risky = self.open_ports & {23, 3389}
        return (not risky, f"ports à risque ouverts : {sorted(risky)}" if risky else "aucun port à risque")


for check in [PasswordPolicyCheck(), FirewallCheck({22, 80, 3389})]:
    ok, detail = check.run()                     # duck typing : seule la méthode run() compte
    print(f"{'✅' if ok else '❌'} {type(check).__name__} : {detail}")

### Exercice 5 ⭐⭐ – Prédire le MRO

Sans exécuter d'abord, prédisez la valeur de `D().describe()` et de `E().describe()`, puis écrivez-les dans `prediction_d` et `prediction_e`.
Exécutez ensuite pour vérifier. Enfin, calculez `mro_e`, la liste des noms de classes du MRO de `E`.

In [ ]:
class Device:
    def describe(self) -> str:
        return "device"


class Wired(Device):
    def describe(self) -> str:
        return "wired+" + super().describe()


class Wireless(Device):
    def describe(self) -> str:
        return "wireless+" + super().describe()


class D(Wired, Wireless):
    pass


class E(Wireless, Wired):
    def describe(self) -> str:
        return "E:" + super().describe()


# TODO : vos prédictions AVANT d'exécuter la ligne print
prediction_d = "..."
prediction_e = "..."
mro_e = ...
print(D().describe(), E().describe(), mro_e)

<details><summary>💡 Indice</summary>

Dans `Wired.describe`, `super()` ne désigne pas forcément `Device` : il désigne la classe **suivante dans le MRO de l'objet**.

</details>

In [ ]:
# ✅ Vérification
assert prediction_d == D().describe()
assert prediction_e == E().describe()
assert mro_e == ["E", "Wireless", "Wired", "Device", "object"]
print("✅ Exercice 5 réussi")

### Exercice 6 ⭐⭐ – Des mixins réutilisables

Écrivez deux mixins :
- `DictMixin` : `to_dict()` renvoie les attributs **publics** de l'objet (ceux dont le nom ne commence pas par `_`) ;
- `ComparableByKeyMixin` : `__eq__` et `__lt__` comparent `self.sort_key()` (méthode que la classe principale doit fournir).

Appliquez-les aux classes `License` et `Contract` fournies.

In [ ]:
# TODO : DictMixin et ComparableByKeyMixin


class License(DictMixin, ComparableByKeyMixin):
    def __init__(self, product: str, seats: int, expires: str) -> None:
        self.product = product
        self.seats = seats
        self.expires = expires          # format AAAA-MM-JJ
        self._key = "XXXX-XXXX"

    def sort_key(self) -> str:
        return self.expires


class Contract(DictMixin, ComparableByKeyMixin):
    def __init__(self, supplier: str, amount: float) -> None:
        self.supplier = supplier
        self.amount = amount

    def sort_key(self) -> float:
        return self.amount


licenses = [License("Office 365", 50, "2027-01-31"), License("Antivirus", 120, "2026-11-30")]
print([l.product for l in sorted(licenses)], licenses[0].to_dict())

<details><summary>💡 Indice</summary>

`vars(self)` renvoie le dictionnaire des attributs de l'objet ; filtrez avec une compréhension.

</details>

In [ ]:
# ✅ Vérification
assert License("A", 1, "2026-01-01").to_dict() == {"product": "A", "seats": 1, "expires": "2026-01-01"}
assert [l.product for l in sorted(licenses)] == ["Antivirus", "Office 365"]
assert Contract("X", 100) == Contract("Y", 100) and Contract("X", 50) < Contract("Y", 100)
assert Contract("X", 100) != License("X", 100, "2026-01-01")
print("✅ Exercice 6 réussi")

### Exercice 7 ⭐⭐ – Exports polymorphes

Trois classes d'export partagent la même interface `export(headers, rows) -> str` :
- `TextExporter` : colonnes alignées à gauche sur 12 caractères, séparées par une espace, **sans** espace final ;
- `CsvExporter(separator=";")` : valeurs séparées par le séparateur ;
- `MarkdownExporter` : tableau Markdown (`| a | b |`, puis `|---|---|`).

Elles héritent d'`Exporter`, qui définit `export` en appelant `format_row` (à redéfinir) pour l'en-tête puis chaque ligne,
et `header_separator()` (vide par défaut, redéfini par `MarkdownExporter`). Construisez ensuite `outputs`, le dictionnaire nom de classe → texte exporté.

In [ ]:
class Exporter:
    """Export générique : le squelette de l'algorithme est commun."""

    def format_row(self, values: list[str]) -> str:
        raise NotImplementedError

    def header_separator(self, column_count: int) -> str | None:
        return None

    def export(self, headers: list[str], rows: list[list]) -> str:
        lines = [self.format_row(headers)]
        separator = self.header_separator(len(headers))
        if separator is not None:
            lines.append(separator)
        lines += [self.format_row([str(v) for v in row]) for row in rows]
        return "\n".join(lines)


# TODO : TextExporter, CsvExporter, MarkdownExporter


headers = ["hostname", "cpu", "ram_gb"]
rows = [["srv-web-01", 8, 32], ["srv-db-01", 16, 128]]
outputs = ...

<details><summary>💡 Indice</summary>

C'est le patron de conception *Template Method* : la classe mère fixe l'algorithme, les filles fournissent les détails.

</details>

In [ ]:
# ✅ Vérification
assert outputs["CsvExporter"] == "hostname;cpu;ram_gb\nsrv-web-01;8;32\nsrv-db-01;16;128"
assert outputs["MarkdownExporter"].splitlines()[:2] == ["| hostname | cpu | ram_gb |", "|---|---|---|"]
assert outputs["TextExporter"].splitlines()[1] == "srv-web-01   8            32"
assert CsvExporter(",").export(["a"], [[1]]) == "a\n1"
print("✅ Exercice 7 réussi")

### Exercice 8 ⭐⭐⭐ – Stratégies de sauvegarde avec une classe abstraite

Écrivez la classe abstraite `BackupStrategy(ABC)` :
- `__init__(self, name: str)` ;
- méthode abstraite `select(files: dict[str, int]) -> list[str]` : `files` associe un chemin à son « jour de dernière modification » ;
- méthode abstraite `retention_days() -> int` ;
- méthode concrète `run(files, today) -> str` : `"<name> : 3 fichier(s), conservation 30 j"`.

Puis deux sous-classes :
- `FullBackup(name)` : tous les fichiers (triés), conservation 90 jours ;
- `IncrementalBackup(name, last_backup_day)` : fichiers modifiés **après** `last_backup_day` (triés), conservation 30 jours.

In [ ]:
from abc import ABC, abstractmethod


# TODO : BackupStrategy, FullBackup, IncrementalBackup


files = {"/etc/nginx.conf": 290, "/var/www/index.html": 293, "/home/amira/rapport.docx": 294, "/opt/app.jar": 280}
for strategy in [FullBackup("Hebdomadaire"), IncrementalBackup("Quotidienne", last_backup_day=292)]:
    print(strategy.run(files, today=294), strategy.select(files))

<details><summary>💡 Indice</summary>

Le corps d'une méthode abstraite peut se limiter à sa docstring. Une sous-classe qui oublie une méthode abstraite ne pourra pas être instanciée.

</details>

In [ ]:
# ✅ Vérification
try:
    BackupStrategy("x")
    raise AssertionError("BackupStrategy doit être abstraite")
except TypeError:
    pass
assert FullBackup("F").run(files, 294) == "F : 4 fichier(s), conservation 90 j"
assert IncrementalBackup("I", 292).select(files) == ["/home/amira/rapport.docx", "/var/www/index.html"]
assert IncrementalBackup("I", 292).run(files, 294) == "I : 2 fichier(s), conservation 30 j"
assert issubclass(IncrementalBackup, BackupStrategy)
print("✅ Exercice 8 réussi")

---
### Exercice de synthèse ⭐⭐⭐ – Contrôles de supervision

Concevez un mini-système de supervision :
- classe abstraite `HealthCheck(ABC)` avec `name`, la méthode abstraite `check() -> tuple[str, str]` (statut parmi `"OK"`, `"WARNING"`, `"CRITICAL"`, et un détail)
  et une méthode concrète `report() -> str` → `"[OK] disk:/var – 62 %"` ;
- `DiskCheck(mount, usage)` : `name = "disk:" + mount` ; WARNING ≥ 80, CRITICAL ≥ 90 ; détail `"62 %"` ;
- `ServiceCheck(service, running)` : `name = "service:" + service` ; OK si `running`, sinon CRITICAL ; détail `"actif"` / `"arrêté"` ;
- `CertificateCheck(domain, days_left)` : `name = "cert:" + domain` ; WARNING < 30 jours, CRITICAL < 7 ; détail `"expire dans 12 j"`.

Enfin, `run_all(checks)` renvoie `(rapports, statut_global)` où le statut global est le **pire** statut. La fonction doit aussi accepter
n'importe quel objet qui possède une méthode `report()` et un attribut `last_status` (duck typing) : voir `LegacyPing`.

In [ ]:
from abc import ABC, abstractmethod

SEVERITY = {"OK": 0, "WARNING": 1, "CRITICAL": 2}


# TODO : HealthCheck, DiskCheck, ServiceCheck, CertificateCheck


class LegacyPing:
    """Ancien contrôle, sans lien d'héritage avec HealthCheck."""

    def __init__(self, host: str, reachable: bool) -> None:
        self.host = host
        self.last_status = "OK" if reachable else "CRITICAL"

    def report(self) -> str:
        return f"[{self.last_status}] ping:{self.host}"


def run_all(checks: list) -> tuple[list[str], str]:
    """Exécute tous les contrôles ; renvoie les rapports et le pire statut."""
    # TODO


checks = [DiskCheck("/var", 62), DiskCheck("/data", 85), ServiceCheck("nginx", True),
          CertificateCheck("intranet.entreprise.tn", 12), LegacyPing("10.0.0.1", True)]
reports, overall = run_all(checks)
print("\n".join(reports), "\nGlobal :", overall)

<details><summary>💡 Indice</summary>

- `report()` appelle `self.check()` : c'est la sous-classe qui décide (polymorphisme).
- Pire statut : `max(statuts, key=lambda s: SEVERITY[s], default="OK")`.

</details>

In [ ]:
# ✅ Vérification
assert reports == ["[OK] disk:/var – 62 %", "[WARNING] disk:/data – 85 %", "[OK] service:nginx – actif",
                   "[WARNING] cert:intranet.entreprise.tn – expire dans 12 j", "[OK] ping:10.0.0.1"]
assert overall == "WARNING"
assert run_all([ServiceCheck("cron", False), LegacyPing("x", True)])[1] == "CRITICAL"
assert run_all([])[1] == "OK"
try:
    HealthCheck("x")
    raise AssertionError("HealthCheck doit être abstraite")
except TypeError:
    pass
print("✅ Exercice de synthèse réussi")

---
## Projet fil rouge – étape 6 : hiérarchie d'équipements

![UML étape 6](img/s09_gestparc.svg)

**À faire :**
1. `Equipment` devient **abstraite** : méthodes abstraites `maintenance_interval_months()` et `specs()` ; la catégorie devient un attribut de classe `CATEGORY` (exposé par la propriété `category`), avec un préfixe de numéro `PREFIX`.
   Ajoutez `has_expected_prefix()` et un `__repr__` qui affiche le **nom de la classe réelle**.
2. Sous-classes `Laptop` (fournie en exemple), `Server`, `Printer`, `NetworkDevice`, avec leurs attributs propres (arguments **nommés obligatoires** après `*`) et un `to_dict()` qui **étend** celui du parent.
3. `NetworkMixin` (fourni) ajoute une adresse IP validée aux serveurs, imprimantes et équipements réseau.
4. `equipment_from_dict(data)` : fabrique qui choisit la bonne sous-classe grâce à `EQUIPMENT_TYPES`.
5. `Inventory` : `by_type(cls)`, `maintenance_plan()` (polymorphisme, trié par périodicité) et `ip_plan()` (équipements ayant le mixin).

In [ ]:
from abc import ABC, abstractmethod


class NetworkMixin:
    """Ajoute une adresse IP gérée à un équipement réseau."""

    _ip_address: str | None = None

    @staticmethod
    def is_valid_ip(ip: str) -> bool:
        """Vérifie le format d'une adresse IPv4."""
        parts = ip.split(".")
        return len(parts) == 4 and all([p.isdigit() and 0 <= int(p) <= 255 for p in parts])

    @property
    def ip_address(self) -> str | None:
        return self._ip_address

    @ip_address.setter
    def ip_address(self, value: str | None) -> None:
        if value is not None and not self.is_valid_ip(value):
            print(f"Adresse IP refusée : {value}")
            return
        self._ip_address = value


class Equipment(ABC):
    """Équipement du parc informatique (classe abstraite)."""

    CATEGORY = "equipment"
    PREFIX = "EQ"
    count = 0

    def __init__(self, asset_tag: str, brand: str, model: str, purchase_year: int,
                 price: float, assigned_to: str | None = None) -> None:
        self.asset_tag = asset_tag.upper()
        self.brand = brand
        self.model = model
        self.purchase_year = purchase_year
        self._price = 0.0
        self.price = price
        self._assigned_to = assigned_to
        Equipment.count += 1

    # --- Méthodes abstraites : chaque sous-classe DOIT les définir
    @abstractmethod
    def maintenance_interval_months(self) -> int:
        """Périodicité de maintenance préventive, en mois."""

    @abstractmethod
    def specs(self) -> str:
        """Caractéristiques techniques propres au type d'équipement."""

    # --- Méthodes communes
    @property
    def category(self) -> str:
        return self.CATEGORY

    @staticmethod
    def is_valid_tag(tag: str) -> bool:
        return len(tag) == 6 and tag[:2].isalpha() and tag[2] == "-" and tag[3:].isdigit()

    def has_expected_prefix(self) -> bool:
        # TODO
        ...

    @property
    def price(self) -> float:
        return self._price

    @price.setter
    def price(self, value: float) -> None:
        if float(value) < 0:
            print(f"Prix refusé pour {self.asset_tag} : {value}")
            return
        self._price = float(value)

    @property
    def assigned_to(self) -> str | None:
        return self._assigned_to

    def age(self, current_year: int) -> int:
        return current_year - self.purchase_year

    def is_available(self) -> bool:
        return self._assigned_to is None

    def assign(self, employee: str) -> bool:
        if not self.is_available():
            return False
        self._assigned_to = employee
        return True

    def unassign(self) -> None:
        self._assigned_to = None

    def to_dict(self) -> dict:
        return {"asset_tag": self.asset_tag, "category": self.category, "brand": self.brand,
                "model": self.model, "purchase_year": self.purchase_year, "price": self.price,
                "assigned_to": self.assigned_to}

    def __str__(self) -> str:
        name = f"{self.brand} {self.model}"
        return (f"{self.asset_tag:<7} | {self.category:<8} | {name:<25} | "
                f"{self.price:>10,.2f} DT | {self.assigned_to or '-'}")

    def __repr__(self) -> str:
        # TODO
        ...

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Equipment) and self.asset_tag == other.asset_tag

    def __lt__(self, other: "Equipment") -> bool:
        return self.asset_tag < other.asset_tag


class Laptop(Equipment):
    """Ordinateur portable."""

    CATEGORY = "laptop"
    PREFIX = "LT"

    def __init__(self, asset_tag: str, brand: str, model: str, purchase_year: int, price: float,
                 assigned_to: str | None = None, *, ram_gb: int = 16, screen_inches: float = 14.0) -> None:
        super().__init__(asset_tag, brand, model, purchase_year, price, assigned_to)
        self.ram_gb = ram_gb
        self.screen_inches = screen_inches

    def maintenance_interval_months(self) -> int:
        return 24

    def specs(self) -> str:
        return f"{self.ram_gb} Go RAM, écran {self.screen_inches}\""

    def to_dict(self) -> dict:
        data = super().to_dict()
        data.update(ram_gb=self.ram_gb, screen_inches=self.screen_inches)
        return data


class Server(NetworkMixin, Equipment):
    """Serveur physique."""
    # TODO : CATEGORY, PREFIX, __init__ (cpu_count, ram_gb, rack, ip_address), méthodes abstraites, to_dict


class Printer(NetworkMixin, Equipment):
    """Imprimante réseau."""
    # TODO : is_color, pages_printed, ip_address ; maintenance tous les 3 mois au-delà de 50 000 pages, sinon 12


class NetworkDevice(NetworkMixin, Equipment):
    """Équipement réseau."""
    # TODO : port_count, ip_address ; maintenance 12 mois


EQUIPMENT_TYPES: dict[str, type[Equipment]] = {
    cls.CATEGORY: cls for cls in (Laptop, Server, Printer, NetworkDevice)
}


def equipment_from_dict(data: dict) -> Equipment:
    """Fabrique : crée l'objet de la bonne sous-classe à partir d'un dictionnaire."""
    # TODO
    ...


class Employee:
    """Employé pouvant recevoir des équipements."""

    def __init__(self, first_name: str, last_name: str, department: str) -> None:
        self.first_name = first_name
        self.last_name = last_name
        self.department = department

    @property
    def full_name(self) -> str:
        return f"{self.first_name} {self.last_name}"

    def __str__(self) -> str:
        return f"{self.full_name} ({self.department})"


class Inventory:
    """Inventaire du parc informatique."""

    def __init__(self) -> None:
        self._items: list[Equipment] = []

    def add(self, equipment: Equipment) -> bool:
        if (not isinstance(equipment, Equipment)
                or not Equipment.is_valid_tag(equipment.asset_tag)
                or equipment in self._items):
            return False
        self._items.append(equipment)
        return True

    def find(self, asset_tag: str) -> Equipment | None:
        tag = asset_tag.upper()
        for equipment in self._items:
            if equipment.asset_tag == tag:
                return equipment
        return None

    def assign(self, asset_tag: str, employee: Employee) -> bool:
        equipment = self.find(asset_tag)
        return equipment is not None and equipment.assign(employee.full_name)

    def by_type(self, cls: type) -> list[Equipment]:
        """Équipements instances d'une classe donnée (sous-classes comprises)."""
        # TODO
        ...

    def stats(self) -> dict[str, int]:
        result: dict[str, int] = {}
        for equipment in self._items:
            result[equipment.category] = result.get(equipment.category, 0) + 1
        return result

    def total_value(self, category: str | None = None) -> float:
        return sum([e.price for e in self._items if category is None or e.category == category])

    def available(self) -> list[Equipment]:
        return [e for e in self._items if e.is_available()]

    def older_than(self, years: int, current_year: int) -> list[Equipment]:
        return [e for e in self._items if e.age(current_year) >= years]

    def maintenance_plan(self) -> list[tuple[str, int]]:
        """Polymorphisme : chaque équipement donne sa propre périodicité."""
        # TODO
        ...

    def ip_plan(self) -> dict[str, str | None]:
        """Adresses IP des équipements réseau (ceux qui ont le mixin)."""
        # TODO
        ...

    def __len__(self) -> int:
        return len(self._items)

    def __contains__(self, item: object) -> bool:
        if isinstance(item, str):
            return self.find(item) is not None
        return item in self._items

    def __getitem__(self, index: int) -> Equipment:
        return self._items[index]


def demo_inventory() -> Inventory:
    """Inventaire de démonstration."""
    inventory = Inventory()
    inventory.add(Server("SV-001", "HPE", "ProLiant DL380", 2019, 18500, cpu_count=32, ram_gb=256,
                         rack="R2", ip_address="10.0.0.10"))
    inventory.add(Laptop("LT-001", "Dell", "Latitude 5440", 2023, 3200, "Amira Ben Salah", ram_gb=32))
    inventory.add(Printer("PR-001", "HP", "LaserJet M404", 2021, 950, pages_printed=62_000,
                          ip_address="10.0.2.50"))
    inventory.add(NetworkDevice("SW-001", "Cisco", "Catalyst 9200", 2020, 4100, port_count=48,
                                ip_address="10.0.0.2"))
    return inventory


inventory = demo_inventory()
for equipment in sorted(inventory):
    print(equipment, "|", equipment.specs())
print(inventory.maintenance_plan())
print(inventory.ip_plan())

In [ ]:
# ✅ Vérification
try:
    Equipment("EQ-001", "X", "Y", 2024, 1)
    raise AssertionError("Equipment doit être abstraite")
except TypeError:
    pass
sv = Server("sv-002", "Dell", "PowerEdge R760", 2024, 21000, cpu_count=48, ip_address="10.0.0.11")
assert sv.category == "server" and sv.has_expected_prefix() and isinstance(sv, NetworkMixin)
assert repr(sv) == "Server('SV-002', 'Dell', 'PowerEdge R760')"
assert sv.specs() == "48 cœurs, 64 Go RAM, baie R1, IP 10.0.0.11"
sv.ip_address = "10.0.0.999"
assert sv.ip_address == "10.0.0.11"
assert not isinstance(Laptop("LT-009", "A", "B", 2024, 1), NetworkMixin)
pr = Printer("PR-009", "Canon", "i-SENSYS", 2024, 800, is_color=True, pages_printed=1000)
assert pr.maintenance_interval_months() == 12 and pr.specs().startswith("couleur, 1,000 pages")
data = sv.to_dict()
assert data["cpu_count"] == 48 and data["category"] == "server"
clone = equipment_from_dict(data)
assert type(clone) is Server and clone == sv and clone.cpu_count == 48 and clone.ip_address == "10.0.0.11"
assert type(equipment_from_dict(NetworkDevice("SW-009", "HPE", "Aruba", 2024, 900, port_count=8).to_dict())) is NetworkDevice
assert [e.asset_tag for e in inventory.by_type(NetworkMixin)] == ["SV-001", "PR-001", "SW-001"]
assert inventory.maintenance_plan() == [("PR-001", 3), ("SV-001", 6), ("SW-001", 12), ("LT-001", 24)]
assert inventory.ip_plan() == {"SV-001": "10.0.0.10", "PR-001": "10.0.2.50", "SW-001": "10.0.0.2"}
assert inventory.stats() == {"server": 1, "laptop": 1, "printer": 1, "network": 1}
print("✅ Étape 6 du projet GestParc validée")

✔️ Version de référence : `scripts/gestparc/main.py` (menu avec détails, plan de maintenance et plan d'adressage IP).

**Prochaine étape (S10)** : les messages « Prix refusé » et les `False` silencieux deviennent des **exceptions métier**.

---
## Pour aller plus loin (bonus)

1. **`__init_subclass__`** : faites en sorte que chaque sous-classe d'`Equipment` s'enregistre automatiquement dans `EQUIPMENT_TYPES` lors de sa définition.
2. **Propriété abstraite** : transformez `CATEGORY` en propriété abstraite (`@property` + `@abstractmethod`). Avantages, inconvénients ?
3. **`dataclasses`** : réécrivez `Employee` avec `@dataclass` et comparez (génère `__init__`, `__repr__`, `__eq__`).

In [ ]:
# À vous